# Sanity Check: candidates_deduped

Basic validation and inspection of the deduplicated candidates dataset.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_parquet('../data/processed/candidates_deduped.parquet')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 7,985 rows × 11 columns


## 1. Schema & Data Types

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7985 entries, 0 to 7984
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   problem_id   7985 non-null   object
 1   problem      7985 non-null   object
 2   answer       7985 non-null   object
 3   answer_type  7985 non-null   object
 4   source       7985 non-null   object
 5   difficulty   7985 non-null   object
 6   split        7985 non-null   object
 7   unit         7985 non-null   object
 8   tolerance    7985 non-null   object
 9   domain       7985 non-null   object
 10  language     7985 non-null   object
dtypes: object(11)
memory usage: 686.3+ KB


In [3]:
# Display dtypes more clearly
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

problem_id     object
problem        object
answer         object
answer_type    object
source         object
difficulty     object
split          object
unit           object
tolerance      object
domain         object
language       object
dtype: object

Memory usage: 8.65 MB


## 2. Nulls & Missing Values

In [4]:
null_counts = df.isnull().sum()
null_pct = 100 * df.isnull().sum() / len(df)

null_summary = pd.DataFrame({
    'column': null_counts.index,
    'null_count': null_counts.values,
    'null_pct': null_pct.values
}).sort_values('null_count', ascending=False)

print(null_summary[null_summary['null_count'] > 0])
if (null_summary['null_count'] == 0).all():
    print("✓ No nulls found!")

Empty DataFrame
Columns: [column, null_count, null_pct]
Index: []
✓ No nulls found!


## 3. Duplicates

In [5]:
dup_count = df.duplicated().sum()
dup_pct = 100 * dup_count / len(df)

print(f"Exact duplicates (all columns): {dup_count} ({dup_pct:.2f}%)")

if dup_count > 0:
    print("\nSample duplicate rows:")
    dup_rows = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)
    print(dup_rows)

Exact duplicates (all columns): 0 (0.00%)


## 4. Basic Statistics

In [6]:
df.describe(include='all').T

,count,unique,top,freq
problem_id,7985,7985,SciBench_RL_00426,1
problem,7985,7985,Suppose that a fair $n$-sided die is rolled $n...,1
answer,7985,6120,\boxed{0},105
answer_type,7985,151,NV,2010
source,7985,5,UGPhysics,5452
difficulty,7985,5,,6067
split,7985,1,train_candidate,7985
unit,7985,829,,5879
tolerance,7985,43,,7869
domain,7985,29,QuantumMechanics,999


## 5. Column-Specific Checks

In [7]:
# String columns: check for obvious issues
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique():,}")
    print(f"  Sample values: {df[col].unique()[:5]}")
    # Check for empty strings
    empty = (df[col] == '').sum()
    if empty > 0:
        print(f"  ⚠ Empty strings: {empty}")


problem_id:
  Unique values: 7,985
  Sample values: ['PHYSICS_00000' 'PHYSICS_00001' 'PHYSICS_00002' 'PHYSICS_00003'
 'PHYSICS_00004']

problem:
  Unique values: 7,985
  Sample values: ['汽化过程。压强为 $1.013 \\times 10^{5} \\mathrm{~Pa}$ 时, 1 mol 的水在 $100^{\\circ} \\mathrm{C}$ 变成水蒸气, 它的内能增加多少？已知在此压强和温度下，水和水蒸气的摩尔体积分别为 $V_{1, \\mathrm{~m}}=18.8 \\mathrm{~cm}^{3} / \\mathrm{mol}$ 和 $V_{\\phi, \\mathrm{m}}=3.01 \\times 10^{4} \\mathrm{~cm}^{3} / \\mathrm{mol}$ ，而水的汽化热 $L=4.06 \\times 10^{4} \\mathrm{~J} / \\mathrm{mol}$ 。'
 'The process of vaporization. At a pressure of $1.013 \\times 10^{5} \\mathrm{~Pa}$, how much does the internal energy of 1 mol of water increase when it turns into steam at $100^{\\circ} \\mathrm{C}$? It is known that at this pressure and temperature, the molar volumes of water and steam are $V_{1, \\mathrm{~m}}=18.8 \\mathrm{~cm}^{3} / \\mathrm{mol}$ and $V_{\\phi, \\mathrm{m}}=3.01 \\times 10^{4} \\mathrm{~cm}^{3} / \\mathrm{mol}$, respectively, and the latent heat of va

## 6. Sample Records (First 10)

In [8]:
df.head(10)

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00000,"汽化过程。压强为 $1.013 \times 10^{5} \mathrm{~Pa}$ 时,...",\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00002,"热水熵变。把 1 kg, 20°C 的水放到 100°C 的炉子上加热, 最后达到 100°...",\boxed{1.01 \times 10^{3}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
3,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,\boxed{1.01 \times 10^{3}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
4,PHYSICS_00004,用白光作光源观察双缝干涉。设缝间距为 $d$ ，试求能观察到的清晰可见光谱的级次。,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
5,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
6,PHYSICS_00006,"在通常亮度下, 人眼瞳孔直径约为 3 mm , 同人眼的最小分辨角是多大? 远处两根细丝之间...",\boxed{2.24 \times 10^{-4}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
7,PHYSICS_00007,"Under normal brightness, the diameter of the h...",\boxed{2.24 \times 10^{-4}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
8,PHYSICS_00008,例 25.7 中如果物体放在凸透镜焦点以内离透镜 15 cm 处，求像的位置及其高度放大倍数。,\boxed{-60},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
9,PHYSICS_00009,Example 25.7: If an object is placed within th...,\boxed{-60},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en


## 7. Sample Records (Random 10)

In [9]:
df.sample(min(10, len(df)))

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
1096,PHYSICS_01157,In the radial direction of a Schwarzschild geo...,\boxed{\mathrm{d}t = \frac{\mathrm{d}r}{V \lef...,Expression,PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Modern Physics,en
7809,SciBench_RL_00104,For the reaction $\mathrm{C}($ graphite $)+\ma...,132.9,numerical,SciBench_RL,,train_candidate,$\mathrm{~kJ} \mathrm{~mol}^{-1}$,,thermo,en
7577,OlympiadBench_OE_TO_physics_en_COMP_00208,"iii. Suppose that in the steady state, conditi...",$I=\frac{9 \epsilon_{0} \mu A V_{0}^{2}}{8 d^{...,Expression,OlympiadBench,,train_candidate,,,OE_TO_physics_en_COMP,en
5631,UGPhysics_SemiconductorPhysics_00146,For an n-type GaAs with a thickness of 0.08 cm...,\boxed{-1.28 \times 10^{-5}},NV,UGPhysics,,train_candidate,\text{m}^3/\text{C},,SemiconductorPhysics,en
3147,UGPhysics_ClassicalElectromagnetism_00323,When a cloud passes over a certain location on...,\boxed{6.7 \times 10^{-4}},NV,UGPhysics,,train_candidate,,,ClassicalElectromagnetism,en
2067,UGPhysics_AtomicPhysics_00152,The physical meaning of the decay constant of ...,\boxed{C},MC,UGPhysics,,train_candidate,,,AtomicPhysics,en
988,PHYSICS_01048,在体积 $V$ 内，$N$ 个电子形成无相互作用的费米气，且温度 $T = 0\mathrm...,\boxed{\varepsilon_{\mathrm{F}} = \frac{h^{2}}...,Expression,PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Modern Physics,en
1746,PHYSICS_01828,How many times larger is the unit size in the ...,2,"[""Numerical"", ""Numerical"", ""Numerical""]",PHYSICS,High School Olympiad,train_candidate,,,Mechanics,en
2185,UGPhysics_AtomicPhysics_00270,Find the threshold energy for gamma ($\gamma$)...,"\boxed{4m_{\mathrm{e}}, 2.044}","NV, NV",UGPhysics,,train_candidate,"None, \mathrm{MeV}",,AtomicPhysics,en
3703,UGPhysics_ClassicalMechanics_00490,A pulse propagates along a taut string:\n\n$$\...,\boxed{-\frac{u}{a}},NV,UGPhysics,,train_candidate,,,ClassicalMechanics,en


# #340 seems inconsistent with answer type? it is one value but the type is a list of 3 values. also need to go through each. same inconsistency for #1408.

In [14]:
df.iloc[1408]['problem']

"计算法布里-珀罗标准具的分辨本领。已知光强反射系数 $R=0.9$，两板之间的距离 $h=2\\mathrm{cm}$，波长 $\\lambda=5000\\mathring{\\mathrm{A}}$。光强分布为：  \n$$\nI_{\\stackrel{\\mathrm{\\tiny~k\\oplus~}}{x\\mathbin{\\vrule}}}/I_{\\mathrm{\\tiny~k\\oplus~}}=1\\bigg/\\Big[1+4\\frac{R}{(1-R)^{2}}\\mathrm{sin}^{2}\\frac{\\delta}{2}\\Big]=1/(1+F\\mathrm{sin}^{2}\\delta/2)\n$$  \n式中 $F = 4R/(1-R)^2$，并定义干涉图样的清晰度为 $V'=\\pi/\\varepsilon$，其中 $\\varepsilon=\\frac{1-R}{\\sqrt{R}}$。分辨本领的公式为：  \n$$\nA = \\frac{\\lambda}{\\updelta\\lambda} = m\\frac{\\pi}{\\upvarepsilon} = m\\frac{\\pi\\sqrt{R}}{1-R}\n$$  \n求分辨本领及对应的谱线宽度分别是多少？"

In [15]:
df[df['source'] == 'PHYSICS']

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00000,"汽化过程。压强为 $1.013 \times 10^{5} \mathrm{~Pa}$ 时,...",\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00002,"热水熵变。把 1 kg, 20°C 的水放到 100°C 的炉子上加热, 最后达到 100°...",\boxed{1.01 \times 10^{3}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
3,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,\boxed{1.01 \times 10^{3}},"[""Numerical"", ""Numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
4,PHYSICS_00004,用白光作光源观察双缝干涉。设缝间距为 $d$ ，试求能观察到的清晰可见光谱的级次。,\boxed{1},Numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
...,...,...,...,...,...,...,...,...,...,...,...
1913,PHYSICS_01995,设一个电路由两个串联的线圈组成，其电感分别为 $L_{1}$ 和 $L_{2}$。线圈之间的...,"[""L_1 + L_2 - 2\\mathfrak{M}"", ""L_1 + L_2 + 2\...","[""Expression"", ""Expression""]",PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en
1914,PHYSICS_01996,A plane-polarized wave falls normally on the s...,\frac{(1-n)^{2}+\kappa^{2}}{(1+n)^{2}+\kappa^{2}},Expression,PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en
1915,PHYSICS_01997,一列平面偏振波垂直入射到具有介电常数 $\varepsilon$ 和导电率 $\sigma^...,\frac{(1-n)^{2}+\kappa^{2}}{(1+n)^{2}+\kappa^{2}},Expression,PHYSICS,Undergraduate/Postgraduate(Physics Major),train_candidate,,,Electromagnetism,en
1916,PHYSICS_01998,A bicycle is cruising in a horizontal motion w...,v(t) = v_{0} e^{-\frac{b t}{m}},"[""Equation"", ""Equation""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Mechanics,en


## 8. Inspect Individual Records (Interactive)

In [11]:
# View a specific record with nice formatting
idx = 0  # Change this to inspect different rows
print(f"\nRecord #{idx}:")
print("=" * 80)
for col, val in df.iloc[idx].items():
    print(f"{col:30s} : {val}")


Record #0:
problem_id                     : PHYSICS_00000
problem                        : 汽化过程。压强为 $1.013 \times 10^{5} \mathrm{~Pa}$ 时, 1 mol 的水在 $100^{\circ} \mathrm{C}$ 变成水蒸气, 它的内能增加多少？已知在此压强和温度下，水和水蒸气的摩尔体积分别为 $V_{1, \mathrm{~m}}=18.8 \mathrm{~cm}^{3} / \mathrm{mol}$ 和 $V_{\phi, \mathrm{m}}=3.01 \times 10^{4} \mathrm{~cm}^{3} / \mathrm{mol}$ ，而水的汽化热 $L=4.06 \times 10^{4} \mathrm{~J} / \mathrm{mol}$ 。
answer                         : \boxed{3.75 \times 10^{4}}
answer_type                    : Numerical
source                         : PHYSICS
difficulty                     : Undergraduate (Non-Physics Major),
split                          : train_candidate
unit                           : 
tolerance                      : 
domain                         : Thermodynamics
language                       : en


## 9. Summary

In [12]:
print(f"✓ Total records: {len(df):,}")
print(f"✓ Total columns: {len(df.columns)}")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")
print(f"✓ Null values: {df.isnull().sum().sum()}")
print(f"✓ Duplicates: {df.duplicated().sum()}")

✓ Total records: 7,985
✓ Total columns: 11
✓ Columns: problem_id, problem, answer, answer_type, source, difficulty, split, unit, tolerance, domain, language
✓ Null values: 0
✓ Duplicates: 0
